In [1]:
from sae_lens import SAE
import os
from transformer_lens import HookedTransformer
import json
import torch


with open('../.credentials.json') as f:
    creds = json.load(f)

os.environ['HF_TOKEN']=creds['HF_TOKEN']

/root/sae/v/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sae, _, _ = SAE.from_pretrained(
    release = "mistral-7b-res-wg", # see other options in sae_lens/pretrained_saes.yaml
    sae_id = "blocks.8.hook_resid_pre", # won't always be a hook point
    device = "cuda"
)   

In [3]:
sae.cfg

SAEConfig(d_in=4096, d_sae=65536, activation_fn_str='relu', apply_b_dec_to_input=False, finetuning_scaling_factor=False, context_size=256, model_name='mistral-7b', hook_name='blocks.8.hook_resid_pre', hook_layer=8, hook_head_index=None, prepend_bos=False, dataset_path='monology/pile-uncopyrighted', dataset_trust_remote_code=True, normalize_activations='constant_norm_rescale', dtype='float32', device='cuda', sae_lens_training_version=None)

In [4]:
model = HookedTransformer.from_pretrained('mistralai/Mistral-7B-v0.1', device='cuda')

/root/sae/v/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards: 100%|██████████| 2/2 [01:25<00:00, 42.92s/it]


Loaded pretrained model mistralai/Mistral-7B-v0.1 into HookedTransformer


In [5]:
input_ids = torch.randint(18276, (2, 128))

attention_mask = torch.ones_like(input_ids)

In [6]:
_, cache = model.run_with_cache(input_ids, attention_mask=attention_mask, prepend_bos=True, stop_at_layer=9)


In [9]:
hidden_state = cache['blocks.8.hook_resid_pre']

In [10]:
features = sae.encode(hidden_state)

In [11]:
features

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]], device='cuda:0',
       grad_fn=<ReluBackward0>)

In [21]:
# print the number of non-zero elements in the features

print((features[0][38] != 0).sum())

tensor(94, device='cuda:0')


In [13]:
print(features.mean())

tensor(0.0028, device='cuda:0', grad_fn=<MeanBackward0>)


In [15]:
print(features.shape)

torch.Size([2, 128, 65536])


In [22]:
features = features.transpose(1, 2)

In [24]:
import torch.nn as nn
dropout = nn.Dropout(0.3)
avg_pool = nn.AvgPool1d(kernel_size=8, stride=8)


In [25]:
features = avg_pool(features)

features = features.transpose(1, 2)

features = features.view(features.shape[0], -1)
features = dropout(features)


In [34]:
fc1 = nn.Linear(sae.cfg.d_sae * int(128/8), 2, bias=False).to('cuda')


In [35]:
features.device

device(type='cuda', index=0)

In [36]:
features.shape

torch.Size([2, 1048576])

In [37]:
x = fc1(features)


In [1]:
import torch

In [4]:
q = torch.rand(1, 128, 65000)

In [3]:
q.save("cruft/q.pt")

NameError: name 'q' is not defined